# BY-V deployment: queued output gathering

Thread 1 captures 30-second windows from the remote BY-V microphone. Thread 2 processes them with one MOSS model on OpenVINO GPU, trims utterance audio from model timestamps, and stores WAV/JSON pairs locally. Timing reports are separate CSV files.

See [README.md](README.md) for installation, output schema, timing details, and recovery behavior. This notebook launches an isolated worker using the shared OpenVINO environment; no model is loaded into the notebook kernel.

All project modules used by this notebook are contained in `speech_to_text_deploy/`.

Edit `settings.json` for shared CLI/notebook defaults. Rerun the setup cell after changing settings.

In [ ]:
from pathlib import Path
import csv
import json
import sys
import importlib

PROJECT_ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (path / "speech_to_text_deploy").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the project.")
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select the project-local .venv kernel.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from speech_to_text_deploy.audio.capture import resolve_microphone
import speech_to_text_deploy.config as deploy_config
deploy_config = importlib.reload(deploy_config)
settings = deploy_config.load_settings()

# Edit settings.json for all model, prompt, audio, VAD, output and timeout settings.
INPUT_DEVICE = settings["input_device"]
SAMPLE_RATE = settings["samplerate"]
INPUT_LATENCY_SECONDS = settings["input_latency_seconds"]
INTERNET_TIME_TIMEOUT_SECONDS = settings["internet_time_timeout_seconds"]
LIVE_AUDIO_ENABLED = settings["live_audio_enabled"]
LIVE_AUDIO_URL = settings["live_audio_url"]
SEGMENT_SECONDS = settings["segment_seconds"]
MAX_AUDIO_SECONDS = settings["max_audio_seconds"]
SOURCE_AUDIO = Path(settings["source_audio"]) if settings["source_audio"] else None
REPLAY_REALTIME = settings["replay_realtime"]

# Model, prompt, VAD and shutdown settings.
MODEL_ID = settings["model_id"]
MODEL_DIR = Path(settings["model_dir"])
OV_DEVICE = settings["ov_device"]
MAX_NEW_TOKENS = settings["max_new_tokens"]
HOTWORDS = tuple(settings["hotwords"])
VAD_THRESHOLD = settings["vad_threshold"]
VAD_MIN_SPEECH_DURATION_MS = settings["vad_min_speech_duration_ms"]
VAD_MIN_SILENCE_DURATION_MS = settings["vad_min_silence_duration_ms"]
VAD_SPEECH_PAD_MS = settings["vad_speech_pad_ms"]
GREETING_PREROLL_KEYWORD = settings["greeting_preroll_keyword"]
GREETING_PREROLL_SECONDS = settings["greeting_preroll_seconds"]
CPU_THREADS = settings["cpu_threads"]
LANGUAGES = tuple(settings["languages"])
MIN_FREE_BYTES = settings["min_free_bytes"]
WORKER_TIMEOUT_SECONDS = settings["continuous_worker_timeout_seconds"]
DRAIN_TIMEOUT_SECONDS = settings["drain_timeout_seconds"]
OUTPUT_BASE = Path(settings["output_base"])

print("Project:", PROJECT_ROOT)
print("Source:", SOURCE_AUDIO if SOURCE_AUDIO is not None else f"BY-V device {resolve_microphone(INPUT_DEVICE)}")
print("Local storage:", OUTPUT_BASE)

Run the next cell to begin collection. It stays busy while recording and draining the queue. Interrupt once to request stop and drain, or create `STOP` inside the printed run directory. The last partial window is retained. The default run captures 120 seconds; compilation and queue drain add to wall-clock time.

The run cell refreshes the lightweight gathering module before creating its configuration, so code updates are picked up in an existing kernel. After changing settings.json, rerun the setup cell to reload the settings; Restart Kernel and Run All also resets imported code and settings.

Model initialization sets the timestamp window from `SEGMENT_SECONDS` (0-30 seconds by default). Each inference prompt also uses the actual file duration, including a shorter final window. Model intervals outside the audio are retained as rejected records; valid clips are saved and later windows continue. A run with rejections reports `success_with_warnings`, not a fully clean success.

In [ ]:
# Refresh orchestration after code updates in a long-lived kernel.
import importlib
import speech_to_text_deploy.config as deploy_config
import speech_to_text_deploy.pipeline as gathering
deploy_config = importlib.reload(deploy_config)
gathering = importlib.reload(gathering)

run_dir = deploy_config.new_run_dir(OUTPUT_BASE)
config = deploy_config.GatheringConfig(
    output_dir=str(run_dir), model_id=MODEL_ID, model_dir=str(MODEL_DIR), ov_device=OV_DEVICE,
    input_device=INPUT_DEVICE, samplerate=SAMPLE_RATE, segment_seconds=SEGMENT_SECONDS,
    input_latency_seconds=INPUT_LATENCY_SECONDS,
    internet_time_timeout_seconds=INTERNET_TIME_TIMEOUT_SECONDS,
    live_audio_enabled=LIVE_AUDIO_ENABLED, live_audio_url=LIVE_AUDIO_URL,
    max_audio_seconds=MAX_AUDIO_SECONDS, max_new_tokens=MAX_NEW_TOKENS,
    hotwords=HOTWORDS, vad_threshold=VAD_THRESHOLD,
    vad_min_speech_duration_ms=VAD_MIN_SPEECH_DURATION_MS,
    vad_min_silence_duration_ms=VAD_MIN_SILENCE_DURATION_MS,
    vad_speech_pad_ms=VAD_SPEECH_PAD_MS,
    greeting_preroll_keyword=GREETING_PREROLL_KEYWORD,
    greeting_preroll_seconds=GREETING_PREROLL_SECONDS,
    cpu_threads=CPU_THREADS, languages=LANGUAGES, min_free_bytes=MIN_FREE_BYTES,
    source_audio=str(SOURCE_AUDIO) if SOURCE_AUDIO is not None else None,
    replay_realtime=REPLAY_REALTIME,
)
print("Run:", run_dir, flush=True)
print("Request stop from another terminal: touch", run_dir / "STOP", flush=True)
try:
    runtime = gathering.gather_isolated(config, timeout_seconds=WORKER_TIMEOUT_SECONDS,
                              drain_timeout_seconds=DRAIN_TIMEOUT_SECONDS)
    print("Processed:", runtime["processed_count"], "windows")
    print("Rejected model intervals:", runtime["rejected_segment_count"])
finally:
    # The worker owns adapter try/finally cleanup; its process exits before return.
    report_path = run_dir / "reports" / "runtime.json"
    if report_path.is_file():
        summary = json.loads(report_path.read_text())
        print("Status:", summary["status"])
        print("OpenVINO GPU memory after unload:", summary.get("ov_memory_after_unload"))
    print("RTF report:", run_dir / "reports" / "segments.csv")

In [ ]:
# Inspect timing separately from model output. This cell does not load a model.
with (run_dir / "reports" / "segments.csv").open(newline="", encoding="utf-8") as file:
    timing_rows = list(csv.DictReader(file))
for row in timing_rows:
    print({key: row[key] for key in ("segment_index", "status", "audio_seconds",
                                     "queue_wait_seconds", "inference_seconds",
                                     "trim_and_store_seconds", "rtf")})
print("Paired transcripts and utterance WAVs:", run_dir / "results")

Optional analysis with pandas (install it into `.venv` with `uv` if needed):

```python
import pandas as pd
metrics = pd.read_csv(run_dir / "reports" / "segments.csv")
completed = metrics.loc[metrics["status"].isin(["success", "partial"])]
display(completed[["segment_index", "queue_wait_seconds", "inference_seconds", "rtf"]])
```

Speaker labels apply independently within each 30-second input window. Estimated UTC capture times and sample-accurate relative offsets are stored with each utterance. An empty parsed result is retained as `no_segments` with its raw transcript. Inspect `reports/runtime.json` for failures and `recordings/` to recover unprocessed inputs.

## Run summary: transcript, timestamped audio and RTF

Run the final cell after collection, including after a failed run with saved reports. It reads the current `run_dir`, shows all reported windows and their RTF, and pairs each transcript with its existing timestamp-trimmed WAV player. RTF is measured per input window; the overall value is duration-weighted over completed windows (success or partial). No model is loaded and no audio is recut.

Rejected model intervals are shown with their original text and times, but have no audio player.


In [ ]:
from IPython.display import HTML, display
import importlib
from speech_to_text_deploy.outputs import report as gathering_report

gathering_report = importlib.reload(gathering_report)
if "run_dir" not in globals():
    raise RuntimeError("Run collection first, or set run_dir to an existing saved run directory.")
display(HTML(gathering_report.gathering_summary_html(run_dir)))
